# BreastMNIST transfer learning starter

**Draft for PGTA review.** Follow the final ELEC0134 brief for assessed requirements. This optional scaffold covers the neural-model component; the classical-model task remains in the brief.

**The data is fixed; the architecture is your choice.** Choose an architecture with suitable pretrained weights and implement `build_model`. No model architecture is supplied as a default.

To investigate pretraining, compare **the same chosen architecture** with random initialisation and the chosen pretrained weights. A model trained from scratch uses random weights and learns on the training split. Neither version may train on validation/test images.


## What you complete

Complete three TODOs: **model construction and binary-head adaptation, one training epoch, and prediction scores**. Data loading, metrics and the experiment loop are supplied.

You may explore multiple models. Explain your final choice and report the relevant trials, including unsuccessful ones. Choose and justify learning rates, augmentation and training budget. The one-epoch default only checks that code runs.

Prepare for a short code walkthrough and one small live modification: explain your reasoning, predict the effect, then run and interpret it. Follow the final brief's AI-use and acknowledgement rules.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
STUDENT_DIR = cwd / 'student' if (cwd / 'student' / 'starter_utils.py').is_file() else cwd
if not (STUDENT_DIR / 'starter_utils.py').is_file():
    raise RuntimeError('Start in the package root or beside this notebook and starter_utils.py.')
PROJECT_ROOT = STUDENT_DIR.parent if (STUDENT_DIR.parent / 'instructor').is_dir() else cwd
DATA_DIR, CACHE_DIR = PROJECT_ROOT / 'data', PROJECT_ROOT / 'cache'
sys.path.insert(0, str(STUDENT_DIR))
print('Working folder:', PROJECT_ROOT)


In [ ]:
import json
from copy import deepcopy
from datetime import datetime, timezone
from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from starter_utils import environment_info, load_official_archive, make_loaders, metrics_from_scores, set_seed

torch.hub.set_dir(str(CACHE_DIR / 'torch'))


## Choose your experiment

Fill in `MODEL_NAME` and `PRETRAINED_WEIGHTS` with your chosen architecture and exact weight identifier/source. Implement construction and loading in TODO 1; the strings document your choice rather than select a supplied model.

Within the pretraining comparison, keep architecture, input processing, split, trainable layers and budget rules consistent across initialisations. Learning rates may be chosen separately on validation data under the same tuning-budget rule.

Complete the TODOs, then enable `RUN_EXPERIMENTS`. Data checks can run before choosing a model.


In [ ]:
MODEL_NAME = ''  # Your chosen architecture; no default model.
PRETRAINED_WEIGHTS = ''  # Exact pretrained weight identifier/source used in build_model.
SEED, THREADS = 42, 4
BATCH_SIZE, IMAGE_SIZE = 16, 224
EPOCHS = 1
AUGMENT = False
TRAINABLE = 'all'  # Optional extension: 'head', with a frozen backbone.
INITIALISATIONS = ['random', 'pretrained']
LEARNING_RATES = {'random': 1e-3, 'pretrained': 1e-3}
RUN_EXPERIMENTS = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED, threads=THREADS)
print('Device:', device)


## Check the data

BreastMNIST is a **binary image-classification** task using 28×28 ultrasound images. Keep the official splits: 546 training, 78 validation, 156 test images.

Original labels are **0 = malignant; 1 = normal/benign**. Malignancy is the positive event, so use softmax column **0**, not column 1. Accuracy alone can hide class imbalance. Follow the teaching team's data-quality policy; do not guess or change clinical labels.


In [ ]:
archive = load_official_archive(DATA_DIR)
for split in ['train', 'val', 'test']:
    labels = archive[f'{split}_labels'].reshape(-1)
    print(split, archive[f'{split}_images'].shape, 'counts [0, 1]:', np.bincount(labels))
counts = np.bincount(archive['train_labels'].reshape(-1))
print('Training majority-class accuracy:', counts.max() / counts.sum())


## Input processing

The default preserves the whole image, resizes it to `IMAGE_SIZE`, repeats grayscale into three channels, and uses ImageNet normalisation. Check that this matches your chosen model. Upsampling adds no medical detail, and transfer from another image domain is not guaranteed to help.

For another input convention, pass your own `train_transform` and `eval_transform` in `LOADER_OPTIONS`. They accept a PIL image and return a tensor. Explain the choice and apply it consistently to both initialisations. Evaluation must be deterministic. The optional default rotation/flip applies only to training images; justify its suitability.


In [ ]:
LOADER_OPTIONS = dict(batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, augment=AUGMENT, seed=SEED)
# For custom models, add train_transform=... and eval_transform=... above if needed.
loaders = make_loaders(DATA_DIR, **LOADER_OPTIONS)
images, labels = next(iter(loaders['train']))
print('Input:', tuple(images.shape), 'labels:', tuple(labels.shape), labels.dtype)


## TODO 1 Build and adapt your chosen model

Construct your architecture, loading the declared weights only for `pretrained` and using random initialisation for `random`. Replace its classification head with **two logits** and document how you found the head and its input width. Check the input convention required by the weights.

For the core comparison, train all layers. For an optional head-only extension, freeze the backbone and leave the new head trainable. Your training function must also keep frozen BatchNorm statistics unchanged; disabling gradients alone does not do this.

Return the model and remove `NotImplementedError`. CrossEntropyLoss expects logits, without a softmax layer. Import the libraries needed by your own implementation.


In [ ]:
def build_model(initialization='pretrained', trainable='all', model_name=None):
    if initialization not in {'random', 'pretrained'} or trainable not in {'all', 'head'}:
        raise ValueError('Use random/pretrained and all/head.')
    # TODO: construct your architecture, use PRETRAINED_WEIGHTS only for pretrained,
    # adapt the head to two logits, configure trainable parameters, and return it.
    raise NotImplementedError('Implement your chosen binary model.')


## TODO 2 Train one epoch

Implement batch-based training using the supplied criterion and optimizer: clear gradients, compute logits/loss, backpropagate and update parameters. Return the **sample-weighted mean loss**, not an unweighted average of batch means.

Use training mode for the all-layer comparison. For a head-only extension, frozen BatchNorm running statistics must remain fixed; disabling gradients alone does not freeze these buffers.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    # TODO: implement one epoch and return mean loss per sample.
    raise NotImplementedError('Complete one training epoch.')


## TODO 3 Return prediction scores

Evaluate without gradient updates and without training-time BatchNorm/dropout behaviour. Return two one-dimensional NumPy arrays in loader order:

- original integer labels;
- malignancy probabilities from softmax(logits, dim=1)[:, 0].

Return probabilities rather than only hard class predictions, so ROC-AUC and threshold analyses remain possible.


In [ ]:
def predict_scores(model, loader, device):
    # TODO: collect labels and class-0 probabilities, then concatenate.
    raise NotImplementedError('Complete label and malignant-score prediction.')


## Run both initialisations

Training updates parameters; minimum validation log loss selects the checkpoint. Each condition starts with the same data-loader seed. The loop saves settings, history, validation scores and metrics to a new run folder. Record all trials, including failures. Use timing to describe actual compute, and acknowledge the external cost of pretraining.


In [ ]:
def validation_log_loss(labels, scores):
    p = np.clip(np.asarray(scores, dtype=np.float64), 1e-12, 1 - 1e-12)
    return float(-np.where(np.asarray(labels) == 0, np.log(p), np.log1p(-p)).mean())


def run_experiment(initialization):
    set_seed(SEED, threads=THREADS)
    loaders = make_loaders(DATA_DIR, **LOADER_OPTIONS)
    model = build_model(initialization, TRAINABLE, MODEL_NAME).to(device)
    optimizer = torch.optim.SGD(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=LEARNING_RATES[initialization], momentum=0.9, weight_decay=1e-4)
    history, best_loss, best_state = [], float('inf'), None
    for epoch in range(1, EPOCHS + 1):
        started = perf_counter()
        train_loss = train_one_epoch(model, loaders['train'], nn.CrossEntropyLoss(), optimizer, device)
        train_seconds = perf_counter() - started
        labels, scores = predict_scores(model, loaders['val'], device)
        val_loss = validation_log_loss(labels, scores)
        metrics = metrics_from_scores(labels, scores)
        history.append(dict(epoch=epoch, train_loss=float(train_loss), val_loss=val_loss,
                            val_balanced_accuracy=metrics['balanced_accuracy'], train_seconds=train_seconds))
        print(initialization, history[-1])
        if val_loss < best_loss:
            best_loss, best_state = val_loss, deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    labels, scores = predict_scores(model, loaders['val'], device)
    return dict(model=model, history=history, labels=labels.tolist(), scores=scores.tolist(),
                validation=metrics_from_scores(labels, scores))


runs = {}
if RUN_EXPERIMENTS:
    if not MODEL_NAME.strip() or not PRETRAINED_WEIGHTS.strip():
        raise ValueError('Specify your chosen architecture and pretrained weight source first.')
    if EPOCHS < 1:
        raise ValueError('EPOCHS must be positive.')
    RUN_DIR = PROJECT_ROOT / 'results' / datetime.now(timezone.utc).strftime('student_%Y%m%dT%H%M%S%fZ')
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    config = dict(model=MODEL_NAME, pretrained_weights=PRETRAINED_WEIGHTS, seed=SEED, epochs=EPOCHS, trainable=TRAINABLE,
                  initialisations=INITIALISATIONS, learning_rates=LEARNING_RATES,
                  inputs={k: str(v) if callable(v) else v for k, v in LOADER_OPTIONS.items()},
                  environment=environment_info())
    (RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    for initialization in INITIALISATIONS:
        runs[initialization] = run_experiment(initialization)
        record = {k: v for k, v in runs[initialization].items() if k != 'model'}
        (RUN_DIR / f'{initialization}.json').write_text(json.dumps(record, indent=2), encoding='utf-8')
    print('Saved:', RUN_DIR)
else:
    print('Complete three TODOs, then enable RUN_EXPERIMENTS.')


In [ ]:
if runs:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for name, run in runs.items():
        history = run['history']
        epochs = [r['epoch'] for r in history]
        for key, style in [('train_loss', 'o-'), ('val_loss', 'x--')]:
            axes[0].plot(epochs, [r[key] for r in history], style, label=f'{name} {key}')
        axes[1].plot(epochs, [r['val_balanced_accuracy'] for r in history], 'o-', label=name)
    axes[0].set(xlabel='Epoch', ylabel='Cross-entropy loss')
    axes[1].set(xlabel='Epoch', ylabel='Validation balanced accuracy')
    for ax in axes:
        ax.legend()
    fig.tight_layout()
    fig.savefig(RUN_DIR / 'learning_curves.png', dpi=150)
    plt.show()


## Predict, change, explain

Practise with saved **validation** scores. Before running the next cell, predict what changing the malignancy threshold will do to sensitivity, specificity and ROC-AUC. Explain why retraining is unnecessary. Then choose a threshold in `DEMO_THRESHOLD` and compare your prediction with the actual result.

For the short walkthrough, the assessor may specify another threshold or an equivalent small change. Explain your own code and a concrete failure case. This assesses understanding, not typing speed or achieving the highest score.

The confusion matrix uses rows=true and columns=predicted in label order [0 malignant, 1 normal/benign]: **[[TP, FN], [FP, TN]]**. Metrics may stay unchanged if no score crosses the new threshold.


In [ ]:
DEMO_THRESHOLD = None  # Make a prediction first, then enter a value between 0 and 1.
if runs and DEMO_THRESHOLD is not None:
    for name, run in runs.items():
        print(name)
        for threshold in [0.5, DEMO_THRESHOLD]:
            metrics = metrics_from_scores(run['labels'], run['scores'], threshold)
            print({k: metrics[k] for k in ['threshold', 'roc_auc', 'malignant_sensitivity',
                                         'malignant_specificity', 'confusion_matrix']})


## Final held-out evaluation

Choose any threshold using validation data and fix the experiment plan before enabling this cell. The final evaluation uses the **complete official test split**. Do not use test outcomes to revise settings or select a more favourable run.

The switches prevent an accidental test and a repeated test in this kernel. They are a workflow aid, not a guarantee of unbiased evaluation. Record why settings and thresholds were fixed.


In [ ]:
EXPERIMENT_PLAN_LOCKED = False
RUN_FINAL_TEST = False
FINAL_THRESHOLD = 0.5  # Fix a validation-only rule before testing.

if RUN_FINAL_TEST:
    if not EXPERIMENT_PLAN_LOCKED or not runs:
        raise RuntimeError('Train/select models and document the fixed plan first.')
    if globals().get('_final_test_ran', False):
        raise RuntimeError('Test already evaluated in this kernel.')
    test_loader = make_loaders(DATA_DIR, **LOADER_OPTIONS, include_test=True)['test']
    final_results = {}
    _final_test_ran = True
    for name, run in runs.items():
        labels, scores = predict_scores(run['model'], test_loader, device)
        final_results[name] = dict(labels=labels.tolist(), scores=scores.tolist(),
                                  metrics=metrics_from_scores(labels, scores, FINAL_THRESHOLD))
    (RUN_DIR / 'final_test.json').write_text(json.dumps(final_results, indent=2), encoding='utf-8')
    print({name: result['metrics'] for name, result in final_results.items()})


## Explain your results and prepare a walkthrough

Explain your model choice and document architectures explored, weights, preprocessing, trainable layers, relevant trials (including failures), and compute. Distinguish a tuning trial from the final fixed comparison; a higher score alone is not a sufficient explanation.

| Initialisation | Budget | Validation ROC-AUC | Balanced accuracy | Malignant sensitivity | Specificity | Compute |
|---|---|---|---|---|---|---|
| Random | | | | | | |
| Pretrained | | | | | | |

Explain failures, class imbalance, overfitting and limits of a small, low-resolution dataset. Repeat seeds if required by the final brief. Training seeds do not establish uncertainty about new patients. MedMNIST is not intended for clinical use; the supplied split does not establish patient-independent evaluation.

For a roughly five-minute walkthrough, be ready to:

1. Explain your model choice and one relevant unsuccessful experiment.
2. Locate the head and trainable parameters, and explain the loss, class-0 score and split usage.
3. Predict, make and interpret one small change using saved validation results.

Acknowledge permitted AI assistance and explain how you checked its output. Exact timing, format and assessment criteria follow the final brief.


## Official references

- [PyTorch transfer-learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [Torchvision models and pretrained weights](https://docs.pytorch.org/vision/stable/models.html)
- [MedMNIST data and code](https://github.com/MedMNIST/MedMNIST)
- [Official BreastMNIST archive](https://zenodo.org/records/10519652/files/breastmnist.npz?download=1)
